In [ ]:
def rename_pth_files(folder_path, base_name="resnet18_epoch_"):
    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter out files with .pth extension
    pth_files = [f for f in files if f.endswith('.pth')]

    # Sort the files numerically based on their current names
    pth_files.sort(key=lambda x: int(os.path.splitext(x)[0]))

    # Rename each file
    for index, file_name in enumerate(pth_files):
        new_name = f"{base_name}{index + 1}.pth"
        old_file_path = os.path.join(folder_path, file_name)
        new_file_path = os.path.join(folder_path, new_name)
        os.rename(old_file_path, new_file_path)
        print(f"Renamed {file_name} to {new_name}")

if __name__ == "__main__":
    folders = ['mixed_r_f_10', 'mixed_r_f_20', 'mixed_r_f_25', 'mixed_r_f_30', 'mixed_r_f_40', 'mixed_r_f_50', 'mixed_r_f_60', 'mixed_r_f_70', 'mixed_r_f_80', 'mixed_r_f_90']
    for folder in folders:
        rename_pth_files(folder)

In [3]:
from src.model.models import get_model

model, _ = get_model(model_name='resnet18', pretrained=False)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
import os
import torch
from src.texture_shape_bias.texture_bias_evaluation import get_cue_conflict_images, CueConflictDataset, \
    get_content_images, ContentDataset, evaluate_model_on_content, evaluate_model_texutre_shape_bias
from torchvision.transforms import transforms

# get cue-conflict dataset
root = '../data/cue_conflict_meadow_f_f'
cue_conflict_path = os.path.join(root, 'output')
content_path = os.path.join(root, 'content')

# prepare cue-conflict dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_paths, shape_labels, texture_labels = get_cue_conflict_images(cue_conflict_path)
cue_conflict_dataset = CueConflictDataset(img_paths, shape_labels, texture_labels, transform)
cue_conflict_dataloader = torch.utils.data.DataLoader(cue_conflict_dataset, batch_size=1, shuffle=True)

# prepare content dataset
img_paths, labels = get_content_images(content_path)
content_dataset = ContentDataset(img_paths, labels, transform)
content_dataloader = torch.utils.data.DataLoader(content_dataset, batch_size=1, shuffle=True)

content_accuracy = evaluate_model_on_content(model, content_dataloader)
shape_decision, texture_decision = evaluate_model_texutre_shape_bias(model, cue_conflict_dataloader)

Testing on Cue-Conflict Dataset: 100%|██████████| 2560/2560 [00:45<00:00, 56.35it/s]


In [6]:
content_accuracy

0.03125

In [8]:
texture_decision

0.03125

In [9]:
shape_decision

0.03125